<a href="https://colab.research.google.com/github/juliancruz98/TEST-Compiladores-SDT-M-quinas-Virtual-Pila/blob/main/Mi_Copia_de_Laboratorio_Compiladores_SDT_Maquina_Virtual_Pila.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guía de Práctica de Laboratorio: Generación de Código Intermedio (SDT) y Máquina Virtual de Pila

**Asignatura:** Programación de Sistemas de Base II  
**Institución:** Universidad Autónoma de Tamaulipas (Facultad de Ingeniería Tampico)  

**Objetivo de la Práctica:** Comprender el ciclo completo de traducción y ejecución de subprogramas en un compilador: desde la emisión de código intermedio de tres direcciones (`param`, `call`, `return`) mediante un Esquema de Traducción Dirigida por la Sintaxis (SDT) con validación de tipos y paso de parámetros, hasta la ejecución real de llamadas recursivas sobre una Máquina Virtual de Pila con gestión dinámica de Marcos de Activación (*Stack Frames*) y Pila de Operandos.

---

## 1. Introducción Teórica y Contexto

En la arquitectura de un compilador, el soporte para subprogramas (funciones y procedimientos) requiere articular dos fases fundamentales:

1. **Traducción a Código Intermedio (Front-End / Middle-End):** Mediante acciones semánticas (SDT), el compilador consulta la **Tabla de Símbolos** para validar la firma de cada invocación, aplicar **coerciones implícitas de tipo** (ej. promoción de `int` a `float`), gestionar el mecanismo de paso de parámetros (**por valor** o **por referencia** `&`) y emitir cuádruplas de tres direcciones (`param`, `call`, `return`).
2. **Entorno de Tiempo de Ejecución (Back-End / Runtime):** Durante la ejecución del bytecode, cada invocación activa crea dinámicamente un **Marco de Activación (*Stack Frame*)** en la **Pila de Llamadas (*Call Stack*)**. Este marco resguarda la dirección de retorno (*Return Address*), el enlace dinámico al marco llamador (*Saved FP*) y el espacio aislado para parámetros y variables locales, permitiendo así la **recursividad** sin colisión de estados.

---

## 2. Código Base Parte I: Generador SDT de Código Intermedio para Subprogramas

*Instrucciones:* Ejecuta esta celda en Google Colab para analizar cómo el traductor SDT emite código de tres direcciones (TAC) para llamadas simples con coerción de tipos, invocaciones anidadas con paso por referencia y sentencias de retorno.

In [1]:
# ==============================================================================
# Laboratorio Práctico Colab
# Tema: Generador SDT de Código Intermedio para Subprogramas (param, call, return)
# ==============================================================================

from typing import List, Dict, Tuple, Optional, Any

# ------------------------------------------------------------------------------
# 1. TABLA DE SÍMBOLOS Y METADATOS DE FUNCIONES Y PARÁMETROS
# ------------------------------------------------------------------------------

class ParameterSymbol:
    """Metadatos de un parámetro formal."""
    def __init__(self, name: str, param_type: str, is_reference: bool = False):
        self.name = name
        self.param_type = param_type
        self.is_reference = is_reference  # True si es paso por referencia (&)

class FunctionSymbol:
    """Metadatos de una función registrada en la Tabla de Símbolos."""
    def __init__(self, name: str, return_type: str, params: List[ParameterSymbol]):
        self.name = name
        self.return_type = return_type
        self.params = params

class SymbolTable:
    """Tabla de símbolos global para validar funciones y variables."""
    def __init__(self):
        self.functions: Dict[str, FunctionSymbol] = {}
        self.variables: Dict[str, str] = {}  # var_name -> data_type

    def add_variable(self, name: str, var_type: str):
        self.variables[name] = var_type

    def add_function(self, name: str, return_type: str, params: List[ParameterSymbol]):
        self.functions[name] = FunctionSymbol(name, return_type, params)

    def lookup_var(self, name: str) -> Optional[str]:
        return self.variables.get(name, None)

    def lookup_func(self, name: str) -> Optional[FunctionSymbol]:
        return self.functions.get(name, None)

# ------------------------------------------------------------------------------
# 2. GENERADOR DE CÓDIGO DE TRES DIRECCIONES (TAC)
# ------------------------------------------------------------------------------

class TACInstruction:
    """Instrucción de tres direcciones para subprogramas."""
    def __init__(self, op: str, arg1: Optional[str] = None,
                 arg2: Optional[str] = None, result: Optional[str] = None):
        self.op = op
        self.arg1 = arg1 if arg1 is not None else ""
        self.arg2 = arg2 if arg2 is not None else ""
        self.result = result if result is not None else ""

    def __str__(self):
        if self.op == "param":
            return f"param {self.arg1}"
        elif self.op == "call":
            if self.result:
                return f"{self.result} = call {self.arg1}, {self.arg2}"
            return f"call {self.arg1}, {self.arg2}"
        elif self.op == "return":
            if self.arg1:
                return f"return {self.arg1}"
            return "return"
        elif self.op == "=":
            return f"{self.result} = {self.arg1}"
        elif self.op == "CAST":
            return f"{self.result} = ({self.arg2}) {self.arg1}"
        else:
            return f"{self.result} = {self.arg1} {self.op} {self.arg2}"

class SDTCodeGenerator:
    """Motor del esquema SDT para emisión de cuádruplas de subprogramas."""
    def __init__(self):
        self.instructions: List[TACInstruction] = []
        self.temp_counter = 0

    def new_temp(self) -> str:
        self.temp_counter += 1
        return f"t{self.temp_counter}"

    def emit(self, op: str, arg1: Optional[str] = None,
             arg2: Optional[str] = None, result: Optional[str] = None) -> TACInstruction:
        inst = TACInstruction(op, arg1, arg2, result)
        self.instructions.append(inst)
        return inst

    def print_code(self, title: str):
        print("\n=================================================================")
        print(f"   CÓDIGO DE TRES DIRECCIONES: {title}")
        print("=================================================================")
        for idx, inst in enumerate(self.instructions, 1):
            print(f"  [{idx:02d}]  {inst}")
        print("=================================================================\n")

# ------------------------------------------------------------------------------
# 3. TRADUCTOR SDT DE INVOCACIONES Y SUBPROGRAMAS
# ------------------------------------------------------------------------------

class SubprogramSDTTranslator:
    """
    Implementa las acciones semánticas para:
    - Validación de firma de parámetros
    - Coerción de tipos implícita en argumentos
    - Generación de param, call y return
    """
    def __init__(self, sym_table: SymbolTable):
        self.sym_table = sym_table
        self.gen = SDTCodeGenerator()

    def translate_binary_op(self, op: str, addr1: str, type1: str,
                            addr2: str, type2: str) -> Tuple[str, str]:
        """Traducción de expresiones aritméticas simples."""
        # Determinar tipo resultante
        res_type = "float" if "float" in (type1, type2) else "int"
        temp = self.gen.new_temp()
        self.gen.emit(op, addr1, addr2, temp)
        return temp, res_type

    def translate_function_call(self, func_name: str,
                                args: List[Tuple[str, str]]) -> Tuple[str, str]:
        """
        Regla SDT para: E -> id ( lista_args )
        args es una lista de tuplas (direccion_argumento, tipo_argumento)
        """
        func_sym = self.sym_table.lookup_func(func_name)
        if not func_sym:
            raise NameError(f"Error Semántico: La función '{func_name}' no ha sido declarada.")

        if len(args) != len(func_sym.params):
            raise TypeError(f"Error Semántico: '{func_name}' requiere {len(func_sym.params)} argumentos, "
                            f"pero se entregaron {len(args)}.")

        # 1. Evaluar y emitir las cuádruplas param para cada argumento
        processed_param_addrs = []
        for idx, (arg_addr, arg_type) in enumerate(args):
            param_formal = func_sym.params[idx]
            final_addr = arg_addr

            # Verificar coerción de tipo (ej. int -> float)
            if arg_type != param_formal.param_type:
                if arg_type == "int" and param_formal.param_type == "float":
                    cast_temp = self.gen.new_temp()
                    self.gen.emit("CAST", arg_addr, "float", cast_temp)
                    final_addr = cast_temp
                    print(f"   [COERCIÓN SDT] Argumento '{arg_addr}' (int) convertido a float en '{cast_temp}'")
                else:
                    raise TypeError(f"Error Semántico: Tipo incompatible en parámetro {idx+1} de '{func_name}'. "
                                    f"Esperado: {param_formal.param_type}, Entregado: {arg_type}")

            # Manejar paso por referencia si aplica
            if param_formal.is_reference:
                ref_temp = f"&{final_addr}"
                processed_param_addrs.append(ref_temp)
            else:
                processed_param_addrs.append(final_addr)

        # 2. Emitir instrucciones 'param' contiguas
        for p_addr in processed_param_addrs:
            self.gen.emit("param", p_addr)

        # 3. Emitir instrucción 'call'
        num_args = str(len(args))
        if func_sym.return_type == "void":
            self.gen.emit("call", func_name, num_args)
            return "", "void"
        else:
            return_temp = self.gen.new_temp()
            self.gen.emit("call", func_name, num_args, return_temp)
            return return_temp, func_sym.return_type

    def translate_return_statement(self, expr_addr: Optional[str] = None):
        """Regla SDT para: S -> return E; o S -> return;"""
        if expr_addr:
            self.gen.emit("return", expr_addr)
        else:
            self.gen.emit("return")

    def translate_assignment(self, var_name: str, expr_addr: str):
        """Regla SDT para: S -> id = E;"""
        self.gen.emit("=", expr_addr, None, var_name)

# ------------------------------------------------------------------------------
# 4. CASOS DE PRUEBA Y DEMOSTRACIÓN EN COLAB
# ------------------------------------------------------------------------------

def run_subprogram_sdt_lab_demo():
    print("=================================================================")
    print("   LABORATORIO COLAB: ESQUEMA SDT PARA SUBPROGRAMAS (MÓDULO 2.3.4)")
    print("=================================================================\n")

    # 1. Configurar Tabla de Símbolos
    sym_table = SymbolTable()
    sym_table.add_variable("a", "int")
    sym_table.add_variable("b", "int")
    sym_table.add_variable("c", "int")
    sym_table.add_variable("y", "int")

    # Registrar función: float f(int x, float y)
    params_f = [
        ParameterSymbol("x", "int", is_reference=False),
        ParameterSymbol("y", "float", is_reference=False)
    ]
    sym_table.add_function("f", "float", params_f)

    # Registrar función: int g(int &p)  [Paso por Referencia]
    params_g = [
        ParameterSymbol("p", "int", is_reference=True)
    ]
    sym_table.add_function("g", "int", params_g)

    print("1. FIRMAS REGISTRADAS EN LA TABLA DE SÍMBOLOS:")
    print("   - Función 'f': (x: int, y: float) -> float")
    print("   - Función 'g': (&p: int) -> int [Paso por Referencia]\n")

    # --------------------------------------------------------------------------
    # CASO 1: Invocación simple con Expresión y Coerción: y = f(a, b + c)
    # --------------------------------------------------------------------------
    print("---> CASO 1: Traducción de 'y = f(a, b + c)' (Con Coerción Int -> Float en Arg 2)")
    sdt1 = SubprogramSDTTranslator(sym_table)

    # 1. Evaluar subexpresión b + c
    arg2_temp, arg2_type = sdt1.translate_binary_op("+", "b", "int", "c", "int")

    # 2. Lista de argumentos para f(a, b + c)
    args_list1 = [("a", "int"), (arg2_temp, arg2_type)]

    # 3. Traducir llamada f(...)
    ret_temp1, ret_type1 = sdt1.translate_function_call("f", args_list1)

    # 4. Asignación a y
    sdt1.translate_assignment("y", ret_temp1)

    sdt1.gen.print_code("Llamada Simple y = f(a, b + c)")

    # --------------------------------------------------------------------------
    # CASO 2: Invocaciones Anidadas: y = f(a, g(b))
    # --------------------------------------------------------------------------
    print("\n---> CASO 2: Traducción de Llamadas Anidadas 'y = f(a, g(b))'")
    sdt2 = SubprogramSDTTranslator(sym_table)

    # 1. Traducir primero la llamada interna g(b) con paso por referencia
    args_g = [("b", "int")]
    ret_g_temp, ret_g_type = sdt2.translate_function_call("g", args_g)

    # 2. Traducir la llamada externa f(a, ret_g_temp)
    args_f = [("a", "int"), (ret_g_temp, ret_g_type)]
    ret_f_temp, ret_f_type = sdt2.translate_function_call("f", args_f)

    # 3. Asignación a y
    sdt2.translate_assignment("y", ret_f_temp)

    sdt2.gen.print_code("Llamadas Anidadas y = f(a, g(b))")

    # --------------------------------------------------------------------------
    # CASO 3: Definición del Cuerpo de una Función con Return
    # --------------------------------------------------------------------------
    print("\n---> CASO 3: Traducción del Cuerpo de una Función con 'return x + 1;'")
    sdt3 = SubprogramSDTTranslator(sym_table)

    # Evaluar expresión de retorno: x + 1
    ret_expr_temp, _ = sdt3.translate_binary_op("+", "x", "int", "1", "int")
    # Emitir return
    sdt3.translate_return_statement(ret_expr_temp)

    sdt3.gen.print_code("Cuerpo de Función con Return")

# Ejecutar el laboratorio
if __name__ == "__main__":
    run_subprogram_sdt_lab_demo()

   LABORATORIO COLAB: ESQUEMA SDT PARA SUBPROGRAMAS (MÓDULO 2.3.4)

1. FIRMAS REGISTRADAS EN LA TABLA DE SÍMBOLOS:
   - Función 'f': (x: int, y: float) -> float
   - Función 'g': (&p: int) -> int [Paso por Referencia]

---> CASO 1: Traducción de 'y = f(a, b + c)' (Con Coerción Int -> Float en Arg 2)
   [COERCIÓN SDT] Argumento 't1' (int) convertido a float en 't2'

   CÓDIGO DE TRES DIRECCIONES: Llamada Simple y = f(a, b + c)
  [01]  t1 = b + c
  [02]  t2 = (float) t1
  [03]  param a
  [04]  param t2
  [05]  t3 = call f, 2
  [06]  y = t3


---> CASO 2: Traducción de Llamadas Anidadas 'y = f(a, g(b))'
   [COERCIÓN SDT] Argumento 't1' (int) convertido a float en 't2'

   CÓDIGO DE TRES DIRECCIONES: Llamadas Anidadas y = f(a, g(b))
  [01]  param &b
  [02]  t1 = call g, 1
  [03]  t2 = (float) t1
  [04]  param a
  [05]  param t2
  [06]  t3 = call f, 2
  [07]  y = t3


---> CASO 3: Traducción del Cuerpo de una Función con 'return x + 1;'

   CÓDIGO DE TRES DIRECCIONES: Cuerpo de Función con 

---
## 3. Código Base Parte II: Máquina Virtual de Pila con Call Stack y Stack Frames

*Instrucciones:* Ejecuta esta celda para observar la traza ciclo a ciclo (*Fetch-Decode-Execute*) de la Máquina Virtual de arquitectura dual (**Pila de Operandos** + **Pila de Llamadas**) resolviendo el cálculo recursivo de `fact(4)`.

In [2]:
# ==============================================================================
# Laboratorio Práctico Colab
# Tema: Máquina Virtual de Pila con Call Stack, Stack Frames y Recursividad
# ==============================================================================

import enum
from typing import List, Dict, Union, Optional, Any

# ------------------------------------------------------------------------------
# 1. REPERTORIO DE INSTRUCCIONES DE LA MÁQUINA VIRTUAL (OPCODES)
# ------------------------------------------------------------------------------

class OpCode(enum.Enum):
    PUSH_CONST = "PUSH_CONST"  # Pasa constante a la Pila de Operandos
    LOAD_LOCAL = "LOAD_LOCAL"   # Carga variable local/parámetro desde el Stack Frame activo
    STORE_LOCAL= "STORE_LOCAL"  # Guarda cima de Pila de Operandos en var local/parámetro
    ADD        = "ADD"         # Suma aritméticamente los dos superiores
    SUB        = "SUB"         # Resta los dos superiores (izq - der)
    MULT       = "MULT"        # Multiplica los dos superiores
    LE         = "LE"          # Compara menor o igual (<=), devuelve 1 o 0
    JMP        = "JMP"         # Salto incondicional a una instrucción
    JMP_FALSE  = "JMP_FALSE"   # Salta si la cima de la Pila de Operandos es 0 (falso)
    CALL       = "CALL"        # Invocación a función: Crea Stack Frame y salta
    RET        = "RET"         # Retorno de función: Destruye Stack Frame y restaura PC
    PRINT      = "PRINT"       # Imprime la cima de la pila de operandos
    HALT       = "HALT"        # Detiene la Máquina Virtual

class Instruction:
    """Instrucción ejecutable de Bytecode (OpCode + Operand1 + Operand2)."""
    def __init__(self, opcode: OpCode, arg1: Any = None, arg2: Any = None):
        self.opcode = opcode
        self.arg1 = arg1
        self.arg2 = arg2

    def __repr__(self):
        args = []
        if self.arg1 is not None: args.append(str(self.arg1))
        if self.arg2 is not None: args.append(str(self.arg2))
        args_str = ", ".join(args)
        return f"{self.opcode.value:<12} {args_str}"

# ------------------------------------------------------------------------------
# 2. ESTRUCTURA DEL MARCO DE ACTIVACIÓN (STACK FRAME)
# ------------------------------------------------------------------------------

class StackFrame:
    """
    Representa un Marco de Activación (Stack Frame) alojado en la Pila de Llamadas.
    Contiene el contexto local completo de la función en ejecución.
    """
    def __init__(self, func_name: str, return_address: int, saved_fp: Optional['StackFrame']):
        self.func_name = func_name            # Nombre de la función (ej. "factorial")
        self.return_address = return_address  # Dirección de retorno (PC al volver)
        self.saved_fp = saved_fp              # Enlace Dinámico (Control Link / Frame anterior)
        self.locals: Dict[str, Any] = {}      # Espacio para parámetros y variables locales

    def set_local(self, name: str, value: Any):
        self.locals[name] = value

    def get_local(self, name: str) -> Any:
        if name not in self.locals:
            raise NameError(f"Error en Runtime: Variable local o parámetro '{name}' no asignado.")
        return self.locals[name]

    def __repr__(self):
        return f"Frame[{self.func_name} | RA:{self.return_address} | Locals:{self.locals}]"

# ------------------------------------------------------------------------------
# 3. MOTOR DE LA MÁQUINA VIRTUAL DE PILA (STACK VIRTUAL MACHINE)
# ------------------------------------------------------------------------------

class VirtualMachine:
    """
    Máquina Virtual con arquitectura dual:
- Pila de Operandos (Operand Stack) para cómputo de expresiones.
- Pila de Llamadas (Call Stack) con Marcos de Activación (Stack Frames).
    """
    def __init__(self):
        self.code: List[Instruction] = []
        self.pc: int = 0                                     # Program Counter (IP)
        self.operand_stack: List[Any] = []                   # Pila de Operandos LIFO
        self.call_stack: List[StackFrame] = []               # Pila de Llamadas (Call Stack)
        self.current_frame: Optional[StackFrame] = None      # Frame Pointer (FP activo)
        self.function_table: Dict[str, int] = {}             # Tabla de Direcciones de Funciones

    def register_function(self, name: str, address: int):
        self.function_table[name] = address

    def load_code(self, code: List[Instruction]):
        self.code = code
        self.pc = 0
        self.operand_stack.clear()
        self.call_stack.clear()
        # Crear Marco Global Inicial (Main Frame)
        main_frame = StackFrame("main", return_address=-1, saved_fp=None)
        self.call_stack.append(main_frame)
        self.current_frame = main_frame

    def step(self) -> bool:
        """Ejecuta un ciclo Fetch-Decode-Execute. Retorna False al encontrar HALT."""
        if self.pc >= len(self.code) or self.pc < 0:
            return False

        inst = self.code[self.pc]
        current_pc = self.pc
        self.pc += 1  # Incrementar PC de forma predeterminada

        op = inst.opcode
        a1 = inst.arg1
        a2 = inst.arg2

        # --- EXECUTE OPCODES ---
        if op == OpCode.PUSH_CONST:
            self.operand_stack.append(a1)

        elif op == OpCode.LOAD_LOCAL:
            val = self.current_frame.get_local(a1)
            self.operand_stack.append(val)

        elif op == OpCode.STORE_LOCAL:
            if not self.operand_stack:
                raise IndexError("Stack Underflow en STORE_LOCAL")
            val = self.operand_stack.pop()
            self.current_frame.set_local(a1, val)

        elif op in (OpCode.ADD, OpCode.SUB, OpCode.MULT, OpCode.LE):
            if len(self.operand_stack) < 2:
                raise IndexError(f"Stack Underflow en operación {op.value}")
            b = self.operand_stack.pop()
            a = self.operand_stack.pop()

            if op == OpCode.ADD: res = a + b
            elif op == OpCode.SUB: res = a - b
            elif op == OpCode.MULT: res = a * b
            elif op == OpCode.LE: res = 1 if a <= b else 0
            self.operand_stack.append(res)

        elif op == OpCode.JMP:
            self.pc = a1

        elif op == OpCode.JMP_FALSE:
            if not self.operand_stack:
                raise IndexError("Stack Underflow en JMP_FALSE")
            cond = self.operand_stack.pop()
            if cond == 0:
                self.pc = a1

        elif op == OpCode.CALL:
            func_name = a1
            num_args = a2
            if func_name not in self.function_table:
                raise NameError(f"Error en Runtime: Función '{func_name}' no registrada.")

            target_address = self.function_table[func_name]

            # 1. Extraer los argumentos de la Pila de Operandos
            args = []
            for _ in range(num_args):
                args.append(self.operand_stack.pop())
            args.reverse()  # Orden de izquierda a derecha

            # 2. Crear nuevo Stack Frame (Almacena Return Address y Enlace Dinámico saved_fp)
            new_frame = StackFrame(func_name, return_address=self.pc, saved_fp=self.current_frame)

            # 3. Asignar parámetros en el nuevo marco (nombres p0, p1...)
            for idx, arg_val in enumerate(args):
                new_frame.set_local(f"p{idx}", arg_val)

            # 4. Apilar en el Call Stack y actualizar FP
            self.call_stack.append(new_frame)
            self.current_frame = new_frame

            # 5. Saltar a la dirección de la función
            self.pc = target_address

        elif op == OpCode.RET:
            # 1. Obtener valor de retorno de la cima de la Pila de Operandos
            return_value = self.operand_stack.pop() if self.operand_stack else None

            # 2. Desapilar el Stack Frame activo
            finished_frame = self.call_stack.pop()

            # 3. Restaurar FP y PC
            self.current_frame = finished_frame.saved_fp
            self.pc = finished_frame.return_address

            # 4. Colocar el valor retornado en la Pila de Operandos del llamador
            if return_value is not None:
                self.operand_stack.append(return_value)

        elif op == OpCode.PRINT:
            val = self.operand_stack.pop()
            print(f"   [SALIDA CONSOLA VM] => {val}")

        elif op == OpCode.HALT:
            return False

        return True

    def run_with_trace(self):
        """Ejecuta el programa completo mostrando la traza de PC, Pila Operandos y Call Stack."""
        print("\n" + "="*80)
        print("          TRAZA DE EJECUCIÓN PASO A PASO EN LA MÁQUINA VIRTUAL")
        print("="*80)
        print(f"{'PC':<4} | {'Instrucción':<20} | {'Pila Operandos':<18} | {'Call Stack (Profundidad)'}")
        print("-" * 80)

        while self.pc < len(self.code) and self.pc >= 0:
            current_pc = self.pc
            inst_str = str(self.code[current_pc])
            stack_str = str(self.operand_stack)
            frames_summary = f"Depth={len(self.call_stack)} [{self.current_frame.func_name}]"

            print(f" {current_pc:<3} | {inst_str:<20} | {stack_str:<18} | {frames_summary}")

            running = self.step()
            if not running:
                break
        print("="*80 + "\n")

# ------------------------------------------------------------------------------
# 4. PRUEBA Y DEMOSTRACIÓN COMPLETA: FACTORIAL RECURSIVO FACT(4)
# ------------------------------------------------------------------------------

def run_vm_lab_demo():
    print("=================================================================")
    print("   LABORATORIO COLAB: MÁQUINA VIRTUAL CON STACK FRAMES (MÓDULO 2.4)")
    print("   Programa a ejecutar: Factorial Recursivo de 4 -> fact(4)")
    print("=================================================================\n")

    vm = VirtualMachine()

    # Registrar la dirección de inicio de la función 'fact' (Dirección PC = 3)
    vm.register_function("fact", 3)

    # Programa en Bytecode:
    # --- BLOQUE MAIN (Instrucciones 0 a 3) ---
    # 0: PUSH_CONST 4
    # 1: CALL fact, 1        (Llama a fact(4))
    # 2: PRINT               (Imprime resultado 24)
    # 3: HALT
    #
    # --- FUNCIÓN FACT(p0) (Instrucciones 4 a 16) ---
    # 4: LOAD_LOCAL p0       (Carga parámetro p0)
    # 5: PUSH_CONST 1
    # 6: LE                  (Evalúa p0 <= 1)
    # 7: JMP_FALSE 10        (Si es falso, salta al caso recursivo en PC 10)
    # 8: PUSH_CONST 1        (Caso Base: Retorna 1)
    # 9: RET
    # 10: LOAD_LOCAL p0      (Caso Recursivo: Carga p0)
    # 11: LOAD_LOCAL p0
    # 12: PUSH_CONST 1
    # 13: SUB                (Calcula p0 - 1)
    # 14: CALL fact, 1       (Llama recursivamente a fact(p0 - 1))
    # 15: MULT               (Multiplica p0 * fact(p0 - 1))
    # 16: RET                (Retorna el producto)

    bytecode_program = [
        # Main
        Instruction(OpCode.PUSH_CONST, 4),      # PC 0
        Instruction(OpCode.CALL, "fact", 1),    # PC 1
        Instruction(OpCode.PRINT),              # PC 2
        Instruction(OpCode.HALT),               # PC 3

        # Función fact(p0)
        Instruction(OpCode.LOAD_LOCAL, "p0"),   # PC 4  (Inicio fact)
        Instruction(OpCode.PUSH_CONST, 1),      # PC 5
        Instruction(OpCode.LE),                 # PC 6  (p0 <= 1)
        Instruction(OpCode.JMP_FALSE, 10),      # PC 7  (Salta si p0 > 1)

        # Caso Base (p0 <= 1)
        Instruction(OpCode.PUSH_CONST, 1),      # PC 8
        Instruction(OpCode.RET),                # PC 9

        # Caso Recursivo (p0 > 1)
        Instruction(OpCode.LOAD_LOCAL, "p0"),   # PC 10
        Instruction(OpCode.LOAD_LOCAL, "p0"),   # PC 11
        Instruction(OpCode.PUSH_CONST, 1),      # PC 12
        Instruction(OpCode.SUB),                # PC 13 (p0 - 1)
        Instruction(OpCode.CALL, "fact", 1),    # PC 14 (fact(p0 - 1))
        Instruction(OpCode.MULT),               # PC 15 (p0 * fact(p0 - 1))
        Instruction(OpCode.RET)                 # PC 16
    ]

    # Re-ajustar la dirección registrada tras armar la lista
    vm.register_function("fact", 4)

    # Cargar y ejecutar con traza interactiva
    vm.load_code(bytecode_program)
    vm.run_with_trace()

    print("=================================================================")
    print("   EJECUCIÓN RECURSIVA CONCLUIDA CON ÉXITO")
    print("   El resultado final del factorial fact(4) es 24.")
    print("=================================================================")

# Ejecutar el laboratorio
if __name__ == "__main__":
    run_vm_lab_demo()

   LABORATORIO COLAB: MÁQUINA VIRTUAL CON STACK FRAMES (MÓDULO 2.4)
   Programa a ejecutar: Factorial Recursivo de 4 -> fact(4)


          TRAZA DE EJECUCIÓN PASO A PASO EN LA MÁQUINA VIRTUAL
PC   | Instrucción          | Pila Operandos     | Call Stack (Profundidad)
--------------------------------------------------------------------------------
 0   | PUSH_CONST   4       | []                 | Depth=1 [main]
 1   | CALL         fact, 1 | [4]                | Depth=1 [main]
 4   | LOAD_LOCAL   p0      | []                 | Depth=2 [fact]
 5   | PUSH_CONST   1       | [4]                | Depth=2 [fact]
 6   | LE                   | [4, 1]             | Depth=2 [fact]
 7   | JMP_FALSE    10      | [0]                | Depth=2 [fact]
 10  | LOAD_LOCAL   p0      | []                 | Depth=2 [fact]
 11  | LOAD_LOCAL   p0      | [4]                | Depth=2 [fact]
 12  | PUSH_CONST   1       | [4, 4]             | Depth=2 [fact]
 13  | SUB                  | [4, 4, 1]          | Depth

---
## 4. Bitácora de Reporte del Alumno(a)

**Nombre del Alumno(a):** Raciel Julian Cabrales Cruz **Matrícula:** A2223330141

### Fase 1: Traducción Dirigida por la Sintaxis y Código de Tres Direcciones.

1. **Registro de Cuádruplas Emitidas:**
- Temporal generado para la coerción implícita (`int` $\rightarrow$ `float`) en el Caso 1: _________  t2
- Número total de instrucciones TAC emitidas en el Caso 2 (`y = f(a, g(b))`): _________ 7

2. **Pregunta de Análisis:** En el **Caso 2** (`y = f(a, g(b))`), observa el orden en que se emiten las instrucciones `call g, 1` y `param a`. ¿Por qué el traductor SDT debe completar primero la invocación interna `g(b)` antes de emitir la secuencia contigua de instrucciones `param` para la función externa `f`? ¿Qué ocurriría en la pila de argumentos si se intercalaran?
> *[Espacio para respuesta del alumno]*

---

### Fase 2: Traza de Ejecución y Marcos de Activación en la Máquina Virtual.

Completa la tabla de inspección del estado de la Máquina Virtual durante la ejecución de `fact(4)`:

| Momento de Ejecución | Profundidad (`Depth`) del Call Stack | Contenido de la Pila de Operandos | Valor de `p0` en el Frame Activo |
| :--- | :--- | :--- | :--- |
| Inicio en `main` (`PC = 0`) | | | N/A |
| Al alcanzar el Caso Base (`fact(1)`, `PC = 8`) | | | |
| Tras ejecutar `MULT` en `fact(2)` (`PC = 16`) | | | |
| Justo antes de ejecutar `PRINT` en `main` (`PC = 2`) | | | N/A |

- **Pregunta de Análisis:** En la instrucción `CALL`, la VM guarda `return_address = self.pc` (cuando `self.pc` ya fue incrementado). Explica qué sucedería con el flujo de control del programa si el `StackFrame` guardara el `PC` *antes* de incrementarlo, o si no existiera el enlace dinámico (`saved_fp`) al ejecutar `RET`.
> *[Espacio para respuesta del alumno]*

---

### Fase 3: Integración Front-End y Back-End (TAC $\rightarrow$ Bytecode de Pila)

- **Pregunta de Análisis:** Compara la instrucción `param` del Código de Tres Direcciones con el mecanismo de paso de argumentos de la Máquina Virtual. ¿Cómo se traduce una secuencia `param arg1; param arg2; t1 = call func, 2` al repertorio de opcodes de la VM (`PUSH_CONST` / `LOAD_LOCAL` + `CALL func, 2`) y por qué `OpCode.CALL` necesita ejecutar `args.reverse()` al desapilar?
> *[Espacio para respuesta del alumno]*